# Pretrain Commutative CNN Encoder

Build or load a broad unlabeled tensor dataset, train the commutative self-supervised objective, and save encoder weights for downstream classification notebooks.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from src.ml import CommutativeCNNClassifier, CommutativeCNNConfig, LossWeightConfig, OptimizationConfig
from src.tensor_utils import build_unlabeled_tensor_dataset, load_unlabeled_tensor_dataset, save_unlabeled_tensor_dataset
from src.notebook_utils import configure_full_dataframe_display, load_compound_image_condition_map_csv
from src.training.reporting import plot_training_history

configure_full_dataframe_display()

In [ ]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y64_x64.pt")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_cnn/encoder_state.pt")

rebuild_unlabeled_dataset = False
selected_mechanisms = None
selected_concentrations = ["high", "mid", "low"]
include_treatments = True
include_controls = True
max_tensors_per_compound = None
max_tensors_total = None

output_size = (20, 5, 64, 64)
only_active = True
normalize_global_drift = True
loess_frac = 0.25
use_cache = True
use_tiff_cache = True

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(6, 8),
    spatial_kernel_size_z=(1, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(12,),
    temporal_st_kernel_sizes=(3,),
    temporal_ts_channels=(8,),
    temporal_ts_kernel_sizes=(5,),
    spatial_agg_channels=(12,),
    spatial_agg_kernel_size_z=(3,),
    spatial_agg_kernel_size_xy=(3,),
    spatial_agg_stride_z=(1,),
    spatial_agg_stride_xy=(1,),
    spatial_agg_pool_kernel_z=(1,),
    spatial_agg_pool_kernel_xy=(2,),
    spatial_agg_pool_stride_z=(1,),
    spatial_agg_pool_stride_xy=(2,),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=8,
    num_prototypes=8,
    dropout=0.5,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=75,
    learning_rate=2e-4,
    weight_decay=3e-3,
    early_stopping_patience=8,
    early_stopping_min_delta=0.0,
    scheduler_patience=3,
    scheduler_factor=0.7,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    consistency_weight=1.0,
    feature_weight=0.05,
    prototype_temperature=0.1,
)

In [ ]:
if rebuild_unlabeled_dataset or not unlabeled_dataset_path.exists():
    condition_df = load_compound_image_condition_map_csv()
    unlabeled_dataset = build_unlabeled_tensor_dataset(
        condition_df=condition_df,
        output_size=output_size,
        selected_mechanisms=selected_mechanisms,
        selected_concentrations=selected_concentrations,
        include_treatments=include_treatments,
        include_controls=include_controls,
        max_tensors_per_compound=max_tensors_per_compound,
        max_tensors_total=max_tensors_total,
        only_active=only_active,
        normalize_global_drift=normalize_global_drift,
        loess_frac=loess_frac,
        use_cache=use_cache,
        use_tiff_cache=use_tiff_cache,
    )
    save_unlabeled_tensor_dataset(unlabeled_dataset, unlabeled_dataset_path)
else:
    unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)

unlabeled_dataset["tensors"].shape, unlabeled_dataset["metadata"].shape

In [ ]:
display(
    unlabeled_dataset["metadata"][["mechanism_of_action", "compound", "condition_kind", "concentration_band"]]
    .value_counts()
    .rename("n_samples")
    .reset_index()
    .head(30)
)

In [ ]:
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(unlabeled_dataset["tensors"])
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path

In [ ]:
model.pretrain_history_.tail()
